# 09 — Geometry Loop (PCGM → Surrogate → Ranking)

Closes the Track 1 design loop:

```
optimizer_results.json (Pareto candidates)
          │
          ▼
geometry/pcgm.py  — pre-mesh guardrail check → hole-pattern → plenum point-cloud
          │
          ▼  (rejected designs stopped here)
MultiHeadMGN (MeshGraphNet) surrogate  — predicts T, TMA, p, U fields
          │
          ▼
Field metrics  — T uniformity index, TMA uniformity index, Euler number
          │
          ▼
Post-prediction guardrail check  — confidence score on predicted physics
          │
          ▼
Ranked candidate table + field visualisations
          │
          ▼
geometry_loop_results.json  — top designs flagged for CFD (OpenFOAM) validation
```

**Note:** The PCGM (Physics-Constrained Geometric Morphogenesis) point-cloud uses
uniform random sampling of the plenum volume, not a CFD mesh. Surrogate predictions
on PCGM meshes are good for ranking but the top designs should be validated with
full OpenFOAM reactingFoam runs (Module 7 — Verification Loop).

**Requires:** Colab A100 GPU, `07_optimizer.ipynb` completed

## 0. Setup

In [ ]:
import subprocess, sys, os

subprocess.run(['pip', 'install', '-q', 'torch_geometric', 'h5py', 'scipy'], check=True)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/Ranaam21/cfd-ald-app.git'
REPO_DIR = '/content/cfd-ald-app'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or 'Repo up to date.')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Setup done.')

## 1. Paths + imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
from pathlib import Path
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from geometry.pcgm import from_optimizer_candidates, generate, GLOBAL_COLS
from geometry.grammar import NozzlePattern
from physics.guardrails import GuardrailEngine, GuardrailBounds
from physics.calculator import euler

assert torch.cuda.is_available(), 'Switch to A100 GPU runtime'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')

DRIVE_BASE   = Path('/content/drive/MyDrive/cfd-ald-app')
MULTIHEAD_PT = DRIVE_BASE / 'checkpoints' / 'multihead' / 'multihead_final.pt'
OPT_JSON     = DRIVE_BASE / 'checkpoints' / 'optimizer' / 'optimizer_results.json'
OUT_DIR      = DRIVE_BASE / 'checkpoints' / 'geometry_loop'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RHO_N2 = 1.145  # kg/m³, N2 (nitrogen) at 300 K

print('Imports OK.')

## 2. Load Pareto candidates from optimizer

Takes the top designs from `07_optimizer.ipynb`.
We run all of them through the PCGM (Physics-Constrained Geometric Morphogenesis)
pipeline; the guardrail engine will reject any that are physically inconsistent
before we spend GPU time on inference.

In [ ]:
with open(OPT_JSON) as f:
    opt_data = json.load(f)

top_candidates = opt_data['top_candidates']
print(f'Loaded {len(top_candidates)} Pareto candidates from optimizer')
print(f'Objectives used: TMA_UI, T_UI, neg_Eu')
print()

# Preview
print(f'{"Rank":>4}  {"D [mm]":>8}  {"pitch/D":>8}  {"Q [slm]":>8}  {"TMA_UI":>10}  {"T_UI":>10}')
print('-' * 60)
for c in top_candidates[:10]:
    print(f"{c['pareto_rank']:>4}  "
          f"{c['D_mm']:>8.2f}  "
          f"{c['pitch_over_D']:>8.2f}  "
          f"{c['Q_slm']:>8.2f}  "
          f"{c.get('TMA_UI', float('nan')):>10.4f}  "
          f"{c.get('T_UI',   float('nan')):>10.4f}")

## 3. PCGM pipeline — pre-mesh guardrail check + point-cloud generation

For each candidate:
1. Compute Re (Reynolds), Ma (Mach), Da (Damköhler), Sc (Schmidt) from design params
2. Check against guardrail bounds — **reject before meshing** if violated
3. Generate hex-pattern nozzle array on 300 mm faceplate
4. Sample 80,000 random points inside the plenum volume

In [ ]:
# Eu (Euler number) bounds relaxed — ALD (Atomic Layer Deposition) showerheads
# have tiny per-hole velocity → large Eu. Keep Re and Da tight.
bounds = GuardrailBounds(
    Re  = (1.0,   5000.0),
    Ma  = (0.0,   0.3),
    Da  = (1e-4,  200.0),
    Eu  = (0.5,   1e10),    # relaxed — large Eu is expected for showerheads
)

print(f'Running PCGM pipeline on {len(top_candidates)} candidates...')
print()

pcgm_results = from_optimizer_candidates(
    top_candidates,
    bounds   = bounds,
    pattern  = NozzlePattern.HEX,
    n_points = 80_000,
)

accepted = [r for r in pcgm_results if r.accepted]
rejected = [r for r in pcgm_results if not r.accepted]

print(f'\nPCGM summary:')
print(f'  Accepted: {len(accepted)} / {len(pcgm_results)}')
print(f'  Rejected: {len(rejected)}')
if rejected:
    for r in rejected:
        print(f'    Rejected: {r.reason[:80]}')

## 4. Load MultiHeadMGN (MeshGraphNet) surrogate

In [ ]:
from torch_geometric.nn import MessagePassing

def mlp(in_dim, out_dim, hidden=None, n_layers=2):
    hidden = hidden or out_dim
    dims = [in_dim] + [hidden] * (n_layers - 1) + [out_dim]
    mods = []
    for i in range(len(dims) - 1):
        mods.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:
            mods.append(nn.SiLU())
    return nn.Sequential(*mods)

class MGNProcessor(MessagePassing):
    def __init__(self, hidden):
        super().__init__(aggr='sum')
        self.edge_mlp  = mlp(3*hidden, hidden)
        self.node_mlp  = mlp(2*hidden, hidden)
        self.edge_norm = nn.LayerNorm(hidden)
        self.node_norm = nn.LayerNorm(hidden)
    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        N = x.size(0)
        new_e = self.edge_norm(self.edge_mlp(torch.cat([x[src], x[dst], edge_attr], -1)) + edge_attr)
        agg   = self.propagate(edge_index, x=x, edge_attr=new_e, size=(N, N))
        new_x = self.node_norm(self.node_mlp(torch.cat([x, agg], -1)) + x)
        return new_x, new_e
    def message(self, edge_attr): return edge_attr

class MultiHeadMGN(nn.Module):
    def __init__(self, node_dim, edge_dim, flow_out=4, heat_out=1, species_out=1, hidden=128, n_layers=8):
        super().__init__()
        self.node_enc    = mlp(node_dim, hidden)
        self.edge_enc    = mlp(edge_dim, hidden)
        self.processors  = nn.ModuleList([MGNProcessor(hidden) for _ in range(n_layers)])
        self.flow_dec    = mlp(hidden, flow_out)
        self.heat_dec    = mlp(hidden, heat_out)
        self.species_dec = mlp(hidden, species_out)
    def forward(self, x, edge_index, edge_attr):
        x = self.node_enc(x)
        e = self.edge_enc(edge_attr)
        for proc in self.processors:
            x, e = proc(x, edge_index, e)
        return self.flow_dec(x), self.heat_dec(x), self.species_dec(x)

ckpt      = torch.load(MULTIHEAD_PT, map_location='cpu')
cfg       = ckpt['cfg']
norm      = ckpt['norm']
node_mean = np.array(norm['node_mean'], dtype=np.float32)
node_std  = np.array(norm['node_std'],  dtype=np.float32)
out_mean  = np.array(norm['out_mean'],  dtype=np.float32)
out_std   = np.array(norm['out_std'],   dtype=np.float32)

model = MultiHeadMGN(
    node_dim    = cfg['node_input_dim'],
    edge_dim    = cfg['edge_input_dim'],
    flow_out    = cfg['flow_out_dim'],
    heat_out    = cfg['heat_out_dim'],
    species_out = cfg['species_out_dim'],
    hidden      = cfg['hidden_dim'],
    n_layers    = cfg['n_layers'],
).to(DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Surrogate loaded ({sum(p.numel() for p in model.parameters())/1e6:.2f}M params)')
print(f'K neighbours per node: {cfg["k_neighbors"]}')

## 5. Run surrogate inference on all accepted PCGM designs

In [ ]:
from tqdm import tqdm

K = cfg['k_neighbors']

def run_surrogate(pcgm_result):
    """Run MultiHeadMGN on a PCGMResult inference_data dict."""
    data         = pcgm_result.inference_data
    coords       = data['coords']          # [N, 3]
    node_feats   = data['node_features']   # [N, 4]
    global_feats = data['global_features'] # [18]

    N  = len(coords)
    gb = np.tile(global_feats, (N, 1))     # broadcast global → [N, 18]
    xi = np.concatenate([node_feats, gb], axis=1).astype(np.float32)  # [N, 22]

    # k-NN (k-nearest neighbours) graph
    tree = cKDTree(coords)
    _, idx = tree.query(coords, k=K+1)
    idx = idx[:, 1:]
    src  = np.repeat(np.arange(N), K)
    dst  = idx.flatten()
    diff = coords[dst] - coords[src]
    dist = np.linalg.norm(diff, axis=1, keepdims=True)
    med  = float(np.median(dist)) + 1e-8
    ef   = np.concatenate([diff / med, dist / med], axis=1).astype(np.float32)

    x  = torch.from_numpy((xi - node_mean) / node_std).to(DEVICE)
    ei = torch.tensor(np.stack([src, dst]), dtype=torch.long).to(DEVICE)
    ea = torch.from_numpy(ef).to(DEVICE)

    with torch.no_grad():
        fp, hp, sp = model(x, ei, ea)
    fp = fp.cpu().numpy()
    hp = hp.cpu().numpy().flatten()
    sp = sp.cpu().numpy().flatten()
    del x, ei, ea
    torch.cuda.empty_cache()

    preds = np.zeros((N, 6), dtype=np.float32)
    preds[:, :4] = fp * out_std[:4] + out_mean[:4]   # Ux, Uy, Uz, p
    preds[:, 4]  = hp * out_std[4]  + out_mean[4]    # T (temperature)
    preds[:, 5]  = sp * out_std[5]  + out_mean[5]    # TMA (trimethylaluminium mass fraction)

    return coords, preds


print(f'Running surrogate on {len(accepted)} accepted designs...')
inference_outputs = []   # list of (pcgm_result, coords, preds)

for pcgm_r in tqdm(accepted):
    coords, preds = run_surrogate(pcgm_r)
    inference_outputs.append((pcgm_r, coords, preds))

print(f'Inference complete: {len(inference_outputs)} designs')

## 6. Compute field metrics + post-prediction guardrail check

For each design, compute:
- **T_UI** (temperature uniformity index) = 1 − CV(T) at bottom 10% of plenum
- **TMA_UI** (TMA mass-fraction uniformity index) at 15–55% height band
- **Eu** (Euler number) from predicted pressure drop
- **Guardrail confidence** on predicted physics

In [ ]:
def uniformity_index(field: np.ndarray) -> float:
    m = float(np.mean(field))
    if abs(m) < 1e-10:
        return float('nan')
    return float(1.0 - np.std(field) / abs(m))


def compute_field_metrics(pcgm_r, coords, preds):
    z     = coords[:, 2]
    z_min = z.min(); z_rng = z.max() - z_min

    # Temperature uniformity index at wafer-side (bottom 10%)
    wm    = z < (z_min + 0.10 * z_rng)
    T_UI  = uniformity_index(preds[wm, 4]) if wm.sum() > 5 else float('nan')

    # TMA (trimethylaluminium) uniformity at mid-domain (15–55%)
    sm    = (z > z_min + 0.15 * z_rng) & (z < z_min + 0.55 * z_rng)
    tma_v = preds[sm, 5]
    if sm.sum() > 5 and tma_v.max() > 1e-6:
        TMA_UI = uniformity_index(tma_v)
    elif sm.sum() > 5:
        TMA_UI = float(-np.std(tma_v))   # fallback: -std (maximise = minimise spread)
    else:
        TMA_UI = float('nan')

    # Euler number from predicted pressure drop
    gf      = pcgm_r.inference_data['global_features']
    gd      = dict(zip(GLOBAL_KEYS, gf))
    Q_m3s   = float(gd['flow_rate_slm']) * 1.667e-5
    D       = float(gd['D'])
    n_h     = max(int(round(float(gd['n_holes']))), 1)
    V_noz   = Q_m3s / (n_h * np.pi * (D / 2) ** 2)
    dp      = float(abs(preds[:, 3].max() - preds[:, 3].min()))
    Eu      = euler(max(dp, 1e-3), RHO_N2, max(V_noz, 1e-3))

    # Post-prediction guardrail check (adds Eu computed from predictions)
    dim_vals = {k: float(gd[k]) for k in ['Re', 'Pr', 'Sc', 'Ma', 'Da', 'Pe_h', 'Pe_m']}
    dim_vals['Eu'] = Eu
    gr_post = GuardrailEngine(GuardrailBounds(Eu=(0.5, 1e10))).check(dim_vals)

    return {
        'T_UI':      T_UI,
        'TMA_UI':    TMA_UI,
        'Eu':        Eu,
        'T_mean':    float(preds[:, 4].mean()),
        'T_range':   float(preds[:, 4].max() - preds[:, 4].min()),
        'TMA_max':   float(preds[:, 5].max()),
        'confidence_post': gr_post.confidence,
    }


GLOBAL_KEYS = [
    'Re','Pr','Sc','Ma','Pe_h','Pe_m','Da',
    'D','pitch_over_D','H_plenum','t_face','standoff',
    'flow_rate_slm','beta','v_th','D_m','n_holes','open_area_frac',
]

# Compute metrics for all accepted designs
records = []
for pcgm_r, coords, preds in inference_outputs:
    gf  = pcgm_r.inference_data['global_features']
    gd  = dict(zip(GLOBAL_KEYS, gf))
    met = compute_field_metrics(pcgm_r, coords, preds)

    records.append({
        'D_mm':         float(gd['D']) * 1e3,
        'pitch_over_D': float(gd['pitch_over_D']),
        'Q_slm':        float(gd['flow_rate_slm']),
        'n_holes':      int(round(float(gd['n_holes']))),
        'Re':           float(gd['Re']),
        'Da':           float(gd['Da']),
        'confidence_pre':  pcgm_r.confidence,
        **met,
        '_pcgm':  pcgm_r,
        '_coords': coords,
        '_preds':  preds,
    })

print(f'Metrics computed for {len(records)} designs')
print()
print(f'{"#":>3}  {"D[mm]":>6}  {"p/D":>5}  {"Q":>5}  '
      f'{"n_holes":>7}  {"T_UI":>8}  {"TMA_UI":>8}  {"conf":>6}')
print('-' * 65)
for i, rec in enumerate(records):
    print(f"{i+1:>3}  {rec['D_mm']:>6.2f}  {rec['pitch_over_D']:>5.1f}  "
          f"{rec['Q_slm']:>5.1f}  {rec['n_holes']:>7}  "
          f"{rec['T_UI']:>8.4f}  {rec['TMA_UI']:>8.4f}  "
          f"{rec['confidence_post']:>6.3f}")

## 7. Rank designs by composite score

Composite score = 0.4 × T_UI_norm + 0.4 × TMA_UI_norm + 0.2 × confidence

All objectives are normalised to [0, 1] across the candidate pool before combining.
Designs with NaN objectives are ranked last.

In [ ]:
import numpy as np

def norm_col(vals):
    arr = np.array(vals, dtype=float)
    valid = arr[~np.isnan(arr)]
    if len(valid) == 0 or valid.max() == valid.min():
        return np.where(np.isnan(arr), 0.0, 0.5)
    lo, hi = valid.min(), valid.max()
    normed = (arr - lo) / (hi - lo)
    normed[np.isnan(arr)] = 0.0
    return normed

T_UI_n   = norm_col([r['T_UI']   for r in records])
TMA_UI_n = norm_col([r['TMA_UI'] for r in records])
conf_n   = norm_col([r['confidence_post'] for r in records])

scores = 0.4 * T_UI_n + 0.4 * TMA_UI_n + 0.2 * conf_n
order  = np.argsort(-scores)   # descending

print('Final ranking (composite score = 0.4×T_UI + 0.4×TMA_UI + 0.2×confidence):')
print()
print(f'{"Rank":>4}  {"D[mm]":>6}  {"p/D":>5}  {"Q[slm]":>7}  '
      f'{"n_holes":>7}  {"Re":>7}  {"T_UI":>8}  {"TMA_UI":>8}  {"score":>7}')
print('-' * 75)

ranked = []
for rank, idx in enumerate(order):
    rec = records[idx]
    print(f"{rank+1:>4}  {rec['D_mm']:>6.2f}  {rec['pitch_over_D']:>5.1f}  "
          f"{rec['Q_slm']:>7.2f}  {rec['n_holes']:>7}  "
          f"{rec['Re']:>7.1f}  "
          f"{rec['T_UI']:>8.4f}  {rec['TMA_UI']:>8.4f}  "
          f"{scores[idx]:>7.4f}")
    ranked.append((rank + 1, rec, scores[idx]))

best_rank, best_rec, best_score = ranked[0]
print(f'\nBest design: D={best_rec["D_mm"]:.2f}mm  pitch/D={best_rec["pitch_over_D"]:.1f}  '
      f'Q={best_rec["Q_slm"]:.2f}slm  score={best_score:.4f}')

## 8. Visualise top 2 designs — field slices at wafer-side of plenum

In [ ]:
def plot_field_slices(rec, rank, z_frac=0.15, n_sample=4000):
    coords = rec['_coords']
    preds  = rec['_preds']
    z = coords[:, 2]
    thresh = z.min() + z_frac * (z.max() - z.min())
    m = z < thresh
    if m.sum() < 20:
        m = np.ones(len(z), dtype=bool)

    xs, ys = coords[m, 0] * 1e3, coords[m, 1] * 1e3

    fields = [
        (preds[m, 4], 'T [K]',       'hot'),
        (preds[m, 5], 'TMA',         'viridis'),
        (preds[m, 3], 'p [m²/s²]',   'coolwarm'),
        (np.linalg.norm(preds[m, :3], axis=1), '|U| [m/s]', 'plasma'),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.suptitle(
        f'Rank {rank} — D={rec["D_mm"]:.1f}mm  pitch/D={rec["pitch_over_D"]:.1f}  '
        f'Q={rec["Q_slm"]:.1f}slm  n={rec["n_holes"]}  score={scores[order[rank-1]]:.4f}',
        fontsize=11
    )

    # Subsample for speed
    idx_s = np.random.default_rng(0).choice(len(xs), min(n_sample, len(xs)), replace=False)

    for ax, (field, label, cmap) in zip(axes, fields):
        sc = ax.scatter(xs[idx_s], ys[idx_s], c=field[idx_s],
                        cmap=cmap, s=3, linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
        ax.set_xlabel('X [mm]'); ax.set_ylabel('Y [mm]')
        ax.set_title(label)
        ax.set_aspect('equal')

    plt.tight_layout()
    out = OUT_DIR / f'rank{rank:02d}_fields.png'
    plt.savefig(str(out), dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')


# Plot top 2 designs
for rank_idx in range(min(2, len(ranked))):
    rank, rec, score = ranked[rank_idx]
    plot_field_slices(rec, rank)

## 9. Save results

Saves `geometry_loop_results.json` with:
- All ranked designs + field metrics
- Top candidates flagged for OpenFOAM (reactingFoam) CFD validation
- PCGM (Physics-Constrained Geometric Morphogenesis) geometry info (n_holes, open area, pitch)

In [ ]:
def _to_py(obj):
    if isinstance(obj, (np.floating, np.integer)): return float(obj)
    if isinstance(obj, dict):  return {k: _to_py(v) for k, v in obj.items() if not k.startswith('_')}
    if isinstance(obj, list):  return [_to_py(i) for i in obj]
    return obj

# Assemble clean output records (strip private _keys)
output_records = []
for rank, rec, score in ranked:
    pcgm_r = rec['_pcgm']
    clean = {
        'rank':         rank,
        'score':        float(score),
        'D_mm':         rec['D_mm'],
        'pitch_over_D': rec['pitch_over_D'],
        'Q_slm':        rec['Q_slm'],
        'n_holes':      rec['n_holes'],
        'open_area_pct': float(pcgm_r.geometry.open_area_frac * 100),
        'Re':           rec['Re'],
        'Da':           rec['Da'],
        'T_UI':         rec['T_UI'],
        'TMA_UI':       rec['TMA_UI'],
        'Eu':           rec['Eu'],
        'T_mean_K':     rec['T_mean'],
        'T_range_K':    rec['T_range'],
        'TMA_max':      rec['TMA_max'],
        'confidence_pre':  rec['confidence_pre'],
        'confidence_post': rec['confidence_post'],
        'flag_for_cfd':    rank <= 3,   # top 3 go to OpenFOAM validation
        'pattern': 'hex',
    }
    output_records.append(_to_py(clean))

output = {
    'n_pareto_in':   len(top_candidates),
    'n_pcgm_accept': len(accepted),
    'n_ranked':      len(ranked),
    'n_cfd_flagged': sum(1 for r in output_records if r['flag_for_cfd']),
    'score_weights': {'T_UI': 0.4, 'TMA_UI': 0.4, 'confidence': 0.2},
    'ranked_designs': output_records,
}

out_path = OUT_DIR / 'geometry_loop_results.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Saved → {out_path}')

# Summary
print(f'\nGeometry loop summary:')
print(f'  Pareto candidates in:  {output["n_pareto_in"]}')
print(f'  Passed pre-mesh check: {output["n_pcgm_accept"]}')
print(f'  Ranked designs:        {output["n_ranked"]}')
print(f'  Flagged for CFD:       {output["n_cfd_flagged"]} (top 3)')
print()
print('Top 3 designs for OpenFOAM (reactingFoam) CFD validation:')
for r in output_records[:3]:
    print(f'  Rank {r["rank"]}: D={r["D_mm"]:.2f}mm  pitch/D={r["pitch_over_D"]:.1f}  '
          f'Q={r["Q_slm"]:.2f}slm  n_holes={r["n_holes"]}  score={r["score"]:.4f}')
print()
print('Ready for Module 7 — Verification Loop (OpenFOAM CFD on top candidates)')